
# 🎬 批次影片推論（雙模型路徑 + 114/114/114 Letterbox + 輸出座標空間切換）

- **雙模型路徑**：同時設定 `PT_MODEL_PATH`（.pt）與 `INT8_MODEL_PATH`（.xml/.bin），用 `USE_MODEL` 選擇。
- **自訂前處理**：`letterbox_114()` 以 **(114,114,114)** 作為底色，將每幀 **resize+pad** 到 `IMG_SIZE`（預設 640）。
- **輸出座標空間**：`OUTPUT_SPACE` 可選：
  - `original`：將模型輸出的 `bbox/keypoints` **還原到原始影像座標**（推薦、與你現有 JSON 一致）。
  - `resized`：保留在 **letterbox 後的畫布座標**（例如 640×640）。


## 1) 參數設定

In [1]:
# ==== 必填參數 ====
# INPUT_DIR       = "medias/train_video/original/normal"
# OUTPUT_DIR      = "outputs/skeletons/YOLO-detect/normal"
INPUT_DIR       = "medias/train_video/original/sitdown"
OUTPUT_DIR      = "outputs/skeletons/YOLO/YOLO-detect/sitdown"

# 兩種模型路徑（請至少填一個存在的路徑）
PT_MODEL_PATH   = "yolov8n.pt"                    # 原模型（.pt）
INT8_MODEL_PATH = "yolov8n_openvino_int8_model/yolov8n.xml"      # 量化模型（.xml，需同資料夾有 .bin）

# 使用哪個模型："pt" 或 "int8"
USE_MODEL       = "pt"

# 任務類型："detect"（物件）或 "pose"（骨架）
TASK            = "detect"

# 取幀設定
TARGET_FPS      = 10

# 推論閾值（僅 detect 使用）
CONF_THRES      = 0.4

# 支援的影片副檔名
VIDEO_EXTS      = (".mp4", ".avi", ".mov", ".mkv")

# === 自訂前處理 ===
PRE_RESIZE      = True             # True：先做 114/114/114 letterbox 到 IMG_SIZE
IMG_SIZE        = 640              # 目標邊長（預設 640）
LETTERBOX_COLOR = (114, 114, 114)  # 填補底色

# === 輸出座標空間 ===
# original：還原到原始影像座標；resized：保留在 letterbox 後的 640 空間
OUTPUT_SPACE    = "original"       # "original" 或 "resized"


## 2) 環境安裝（第一次使用或新環境才需要）

In [2]:
# 若環境已安裝可跳過
# !pip install ultralytics openvino opencv-python pillow numpy


## 3) 匯入與工具函式（含 letterbox_114、座標映射 與 檢查）

In [3]:
import os, json, time
from pathlib import Path
import numpy as np
import cv2

from ultralytics import YOLO

try:
    import openvino as ov  # 僅在使用 INT8 時需要
except Exception:
    ov = None

def ensure_dir(p: str | Path):
    Path(p).mkdir(parents=True, exist_ok=True)

def normalize_fps(raw_fps: float) -> int:
    candidates = [24, 25, 30, 50, 60]
    for c in candidates:
        if abs(raw_fps - c) <= 2:
            return c
    return int(round(raw_fps))

def compute_frame_interval(orig_fps: float, target_fps: float) -> int:
    return max(int(round(orig_fps / max(target_fps, 1))), 1)

def iter_video_frames(video_path: str | Path, target_fps: float):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"❌ 無法開啟影片：{video_path}")
    raw_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fps_used = normalize_fps(raw_fps)
    interval = compute_frame_interval(fps_used, target_fps)

    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            yield idx, frame, {"orig_fps": raw_fps, "fps_used": fps_used, "interval": interval}
        idx += 1
    cap.release()

def to_int_xyxy(xyxy_tensor_or_list):
    # 同時相容 torch.Tensor / numpy / list
    try:
        import torch
        if isinstance(xyxy_tensor_or_list, torch.Tensor):
            xyxy_tensor_or_list = (
                xyxy_tensor_or_list.detach().cpu().float().numpy()
            )
    except Exception:
        pass
    arr = np.array(xyxy_tensor_or_list, dtype=np.float32).reshape(-1)  # -> [x1,y1,x2,y2]
    return [int(round(v)) for v in arr]


def letterbox_114(img_bgr: np.ndarray, new_shape=(640, 640), color=(114,114,114)):
    # 參考 YOLOv8 的 letterbox 寫法，固定填補色為 114/114/114
    h0, w0 = img_bgr.shape[:2]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)
    new_h, new_w = new_shape

    r = min(new_w / w0, new_h / h0)
    w1, h1 = int(round(w0 * r)), int(round(h0 * r))  # 未填補前的新尺寸

    pad_w = new_w - w1
    pad_h = new_h - h1
    left = pad_w / 2
    top  = pad_h / 2

    # resize
    if (w0, h0) != (w1, h1):
        img_bgr = cv2.resize(img_bgr, (w1, h1), interpolation=cv2.INTER_LINEAR)

    # pad
    top_i    = int(round(top - 0.1))
    left_i   = int(round(left - 0.1))
    img_bgr = cv2.copyMakeBorder(img_bgr, top_i, pad_h - top_i, left_i, pad_w - left_i,
                                 cv2.BORDER_CONSTANT, value=color)

    meta = {
        "r": r,
        "left": left,
        "top": top,
        "orig_hw": (h0, w0),
        "new_hw": (new_h, new_w),
        "scaled_hw": (h1, w1)
    }
    return img_bgr, meta

def map_xyxy_back_to_original(xyxy, meta):
    r = meta["r"]
    left = meta["left"]
    top  = meta["top"]
    H0, W0 = meta["orig_hw"]
    x1, y1, x2, y2 = map(float, xyxy)
    x1 = (x1 - left) / max(r, 1e-6)
    x2 = (x2 - left) / max(r, 1e-6)
    y1 = (y1 - top)  / max(r, 1e-6)
    y2 = (y2 - top)  / max(r, 1e-6)
    x1 = min(max(x1, 0), W0)
    x2 = min(max(x2, 0), W0)
    y1 = min(max(y1, 0), H0)
    y2 = min(max(y2, 0), H0)
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]

def map_kpts_back_to_original(kpts_xy, meta):
    r = meta["r"]
    left = meta["left"]
    top  = meta["top"]
    H0, W0 = meta["orig_hw"]
    out = []
    for x, y in kpts_xy:
        xo = (float(x) - left) / max(r, 1e-6)
        yo = (float(y) - top)  / max(r, 1e-6)
        xo = min(max(xo, 0), W0)
        yo = min(max(yo, 0), H0)
        out.append([float(xo), float(yo)])
    return out

# OUTPUT_SPACE 檢查與警告
__warned_resized_no_preresize = False
def validate_output_space(output_space: str, pre_resize: bool):
    global __warned_resized_no_preresize
    output_space = (output_space or "").strip().lower()
    if output_space not in {"original", "resized"}:
        raise ValueError("OUTPUT_SPACE 只能是 'original' 或 'resized'")
    if output_space == "resized" and not pre_resize and not __warned_resized_no_preresize:
        print("[警告] OUTPUT_SPACE='resized' 但 PRE_RESIZE=False：此時 'resized' 等同於原始幀大小，建議將 PRE_RESIZE=True 以獲得固定 640 空間。")
        __warned_resized_no_preresize = True
    return output_space

def resolve_model_path(use_model: str, pt_path: str | Path, int8_path: str | Path) -> Path:
    use_model = (use_model or "").strip().lower()
    if use_model not in {"pt", "int8"}:
        raise ValueError("USE_MODEL 只能是 'pt' 或 'int8'")
    if use_model == "pt":
        p = Path(pt_path)
        if not p.exists():
            raise FileNotFoundError(f"找不到 PT 模型：{p}")
        if p.suffix.lower() != ".pt":
            raise ValueError(f"PT_MODEL_PATH 必須是 .pt 檔：{p}")
        return p
    else:
        p = Path(int8_path)
        if not p.exists():
            raise FileNotFoundError(f"找不到 INT8 模型(.xml)：{p}")
        if p.suffix.lower() != ".xml":
            raise ValueError(f"INT8_MODEL_PATH 必須是 .xml 檔：{p}")
        bin_path = p.with_suffix(".bin")
        if not bin_path.exists():
            raise FileNotFoundError(f"缺少對應的 .bin：{bin_path}")
        return p


## 4) 載入模型（自動處理 `.pt` 或 `.xml/.bin`，並可對齊 `IMG_SIZE`）

In [4]:
class PredictorWrapper:
    def __init__(self, model, names, mode, device="AUTO"):
        self.model  = model
        self.names  = names
        self.mode   = mode
        self.device = device
    def __call__(self, image_bgr):
        return self.model(image_bgr)[0]

def load_model(model_path: str | Path, task: str, img_size: int = 640) -> PredictorWrapper:
    model_path = Path(model_path)
    suffix = model_path.suffix.lower()

    if suffix == ".pt":
        yolo = YOLO(str(model_path))
        yolo.overrides["imgsz"] = img_size  # 讓內部前處理對齊大小
        names = yolo.model.names
        return PredictorWrapper(yolo, names, task)

    if suffix == ".xml":
        if ov is None:
            raise ImportError("未偵測到 OpenVINO（import openvino 失敗）。請先安裝 openvino 套件後再執行。")
        core = ov.Core()
        ov_model = core.read_model(str(model_path))
        device = "AUTO"
        ov_config = {}
        try:
            ov_model.reshape({0: [1, 3, img_size, img_size]})  # 嘗試對齊 IMG_SIZE
        except Exception:
            pass
        if "GPU" in device or ("AUTO" in device and "GPU" in core.available_devices):
            ov_config = {"GPU_DISABLE_WINOGRAD_CONVOLUTION": "YES"}
        compiled = core.compile_model(ov_model, device, ov_config)

        yolo = YOLO(str(model_path.parent), task=task)
        yolo.overrides["imgsz"] = img_size
        if yolo.predictor is None:
            custom = {"conf": 0.25, "batch": 1, "save": False, "mode": "predict"}
            args = {**yolo.overrides, **custom}
            yolo.predictor = yolo._smart_load("predictor")(overrides=args, _callbacks=yolo.callbacks)
            yolo.predictor.setup_model(model=yolo.model)
        yolo.predictor.model.ov_compiled_model = compiled
        names = yolo.model.names
        return PredictorWrapper(yolo, names, task, device=device)

    raise ValueError(f"不支援的模型副檔名：{suffix}（只支援 .pt 與 .xml）")


## 5) 單幀推論與結果轉換（支援 OUTPUT_SPACE）

In [5]:
def infer_detect_frame(wrapper: PredictorWrapper, frame_bgr, conf_thres=0.3,
                       pre_resize=True, img_size=640, color=(114,114,114), output_space="original"):
    output_space = validate_output_space(output_space, pre_resize)
    img = frame_bgr
    meta = None
    if pre_resize:
        img, meta = letterbox_114(frame_bgr, new_shape=(img_size, img_size), color=color)

    r0 = wrapper(img)

    result = {"objects": []}
    for box in r0.boxes:
        conf = float(box.conf[0])
        if conf < conf_thres:
            continue
        class_id = int(box.cls[0])
        class_name = wrapper.names[class_id]
        xyxy = to_int_xyxy(box.xyxy[0])
        if output_space == "original" and meta is not None:
            xyxy = map_xyxy_back_to_original(xyxy, meta)
        # 若 output_space=="resized"，則保留在 letterbox 後的 640 空間
        result["objects"].append({
            "class_id": class_id,
            "class_name": class_name,
            "confidence": round(conf, 4),
            "bbox": xyxy
        })
    return result

def infer_pose_frame(wrapper: PredictorWrapper, frame_bgr,
                     pre_resize=True, img_size=640, color=(114,114,114), output_space="original"):
    output_space = validate_output_space(output_space, pre_resize)
    img = frame_bgr
    meta = None
    if pre_resize:
        img, meta = letterbox_114(frame_bgr, new_shape=(img_size, img_size), color=color)

    r0 = wrapper(img)

    if output_space == "resized" or meta is None:
        # 直接輸出模型空間（letterbox 後的畫布）或沒做前處理時的空間
        boxes = r0.boxes.xyxy.tolist() if hasattr(r0, "boxes") else []
        keypoints = r0.keypoints.xy.tolist() if hasattr(r0, "keypoints") else []
        return {"boxes": boxes, "keypoints": keypoints}

    # output_space=="original" 且 meta 存在：需要把 boxes / keypoints 還原到原始座標
    boxes_proc = r0.boxes.xyxy.tolist() if hasattr(r0, "boxes") else []
    boxes_out = [map_xyxy_back_to_original(b, meta) for b in boxes_proc]

    kpts_all = r0.keypoints.xy.tolist() if hasattr(r0, "keypoints") else []
    kpts_out = [map_kpts_back_to_original(klist, meta) for klist in kpts_all]

    return {"boxes": boxes_out, "keypoints": kpts_out}


## 6) 批次處理資料夾內所有影片並輸出 JSON

In [9]:
def process_to_json(input_dir, output_dir, model_path, task="detect", target_fps=10, conf_thres=0.3,
                          video_exts=(".mp4",".avi",".mov",".mkv"), pre_resize=True, img_size=640,
                          color=(114,114,114), output_space="original"):
    input_dir  = Path(input_dir)
    output_dir = Path(output_dir)
    ensure_dir(output_dir)

    wrapper = load_model(model_path, task=task, img_size=img_size)

    video_list = [p for p in sorted(input_dir.iterdir()) if p.suffix.lower() in video_exts]
    if not video_list:
        print(f"⚠️ {input_dir} 沒找到影片（副檔名：{video_exts}）")
        return

    print(f"📁 共 {len(video_list)} 支影片待處理；模型：{model_path}（task={task}，OUTPUT_SPACE={output_space}，PRE_RESIZE={pre_resize}，IMG_SIZE={img_size})")

    for vp in video_list:
        results_list = []
        last_time = time.time()
        count = 0

        for idx, frame, info in iter_video_frames(vp, target_fps=target_fps):
            if task == "detect":
                frame_result = infer_detect_frame(wrapper, frame, conf_thres=conf_thres,
                                                 pre_resize=pre_resize, img_size=img_size,
                                                 color=color, output_space=output_space)
                results_list.append(frame_result)
                fps = 1.0 / max(time.time() - last_time, 1e-6)
                last_time = time.time()
                print(f"[{vp.name} | frame {idx:6d}] objects={len(frame_result['objects']):2d} | FPS={fps:5.2f}")
            else:
                frame_result = infer_pose_frame(wrapper, frame,
                                                pre_resize=pre_resize, img_size=img_size,
                                                color=color, output_space=output_space)
                results_list.append(frame_result)
                fps = 1.0 / max(time.time() - last_time, 1e-6)
                last_time = time.time()
                print(f"[{vp.name} | frame {idx:6d}] boxes={len(frame_result['boxes']):2d} | FPS={fps:5.2f}")

            count += 1

        out_path = output_dir / f"{vp.stem}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results_list, f, indent=2, ensure_ascii=False)
        print(f"✅ 完成：{vp.name} → {out_path}（擷取幀數：{count}）\n")


## 7) 執行

In [14]:
# 依 USE_MODEL 解析實際要使用的模型路徑
MODEL_PATH = resolve_model_path(USE_MODEL, PT_MODEL_PATH, INT8_MODEL_PATH)
print(f"使用模型類型：{USE_MODEL} -> {MODEL_PATH}")

process_to_json(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model_path=MODEL_PATH,
    task=TASK,
    target_fps=TARGET_FPS,
    conf_thres=CONF_THRES,
    video_exts=VIDEO_EXTS,
    pre_resize=PRE_RESIZE,
    img_size=IMG_SIZE,
    color=LETTERBOX_COLOR,
    output_space=OUTPUT_SPACE
)

使用模型類型：pt -> yolov8m-pose.pt
📁 共 132 支影片待處理；模型：yolov8m-pose.pt（task=pose，OUTPUT_SPACE=original，PRE_RESIZE=True，IMG_SIZE=640)

0: 640x640 1 person, 35.1ms
Speed: 4.2ms preprocess, 35.1ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)
[Coffee_room_video (1)_part02_clip01.avi | frame      0] boxes= 1 | FPS= 1.28

0: 640x640 1 person, 13.3ms
Speed: 2.6ms preprocess, 13.3ms inference, 3.9ms postprocess per image at shape (1, 3, 640, 640)
[Coffee_room_video (1)_part02_clip01.avi | frame      2] boxes= 1 | FPS=39.93

0: 640x640 1 person, 13.3ms
Speed: 1.8ms preprocess, 13.3ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 640)
[Coffee_room_video (1)_part02_clip01.avi | frame      4] boxes= 1 | FPS=42.25

0: 640x640 1 person, 13.3ms
Speed: 1.7ms preprocess, 13.3ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)
[Coffee_room_video (1)_part02_clip01.avi | frame      6] boxes= 1 | FPS=45.73

0: 640x640 1 person, 13.6ms
Speed: 1.7ms preprocess, 13.6m

In [13]:
# ==== 必填參數 ====
INPUT_DIR       = "medias/train_video/binary/fall"
OUTPUT_DIR      = "outputs/skeletons/binary/YOLO-pose/fall"
CNAME           = "fall"
TASK            = "pose"                                          # 任務類型："detect"（物件）或 "pose"（骨架）
# INPUT_DIR       = f"medias/train_video/binary/{CNAME}"
# OUTPUT_DIR      = f"outputs/skeletons/multi/YOLO-{TASK}/{CNAME}"

# 兩種模型路徑（請至少填一個存在的路徑）
PT_MODEL_PATH   = f"yolov8m{'-pose' if TASK == 'pose' else ''}.pt"  # 原模型（.pt）                             # 原模型（.pt）
INT8_MODEL_PATH = f"yolov8n{'-pose' if TASK == 'pose' else ''}_openvino_int8_model/yolov8n{'-pose' if TASK == 'pose' else ''}.xml"         # 量化模型（.xml，需同資料夾有 .bin）

# 使用哪個模型："pt" 或 "int8"
USE_MODEL       = "pt"

# 取幀設定
TARGET_FPS      = 10

# 推論閾值（僅 detect 使用）
CONF_THRES      = 0.4

# 支援的影片副檔名
VIDEO_EXTS      = (".mp4", ".avi", ".mov", ".mkv")

# === 自訂前處理 ===
PRE_RESIZE      = True             # True：先做 114/114/114 letterbox 到 IMG_SIZE
IMG_SIZE        = 640              # 目標邊長（預設 640）
LETTERBOX_COLOR = (114, 114, 114)  # 填補底色

# === 輸出座標空間 ===
# original：還原到原始影像座標；resized：保留在 letterbox 後的 640 空間
OUTPUT_SPACE    = "original"       # "original" 或 "resized"


### 備註
- **OUTPUT_SPACE**：
  - `original`（預設）：輸出還原到原始影像座標（與你現有 JSON 一致）。
  - `resized`：輸出保留在 letterbox 後的 640 空間；**建議搭配 `PRE_RESIZE=True`**。
- **OpenVINO 模型大小**：會嘗試將 `.xml` reshape 到 `IMG_SIZE`。若 reshape 不支援，請把 `IMG_SIZE` 設為模型原輸入大小（例如 640）。
- **效能小技巧**：關閉 `座標還原`（改用 `resized`）會省去一次座標轉換，會稍快一點。
